In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import json

from qulacs import QuantumCircuit, QuantumState, Observable, NoiseSimulator
from qulacs import state
import numpy as np
from qulacs import QuantumState, DensityMatrix, QuantumCircuit
import numpy as np
import matplotlib.pyplot as plt
######################################
from quri_parts import *
# ==========================================
# 1. Dynamic Path Setup
# ==========================================
notebook_dir = Path.cwd()

# Walk up directories until we find the folder that actually contains 'src'
project_root = None
for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "src").is_dir():
        project_root = parent
        break

if project_root is None:
    # Fallback: Hardcoded absolute path based on your traceback if dynamic search fails
    project_root = Path("~/Desktop/My Folder/indirect-zne").expanduser()

# Add the project root to sys.path so 'src' is recognized as a package
if str(project_root) not in sys.path:
    sys.path.append(str(project_root.resolve()))

# ==========================================
# 2. Local Package Imports via 'src'
# ==========================================
# Importing utilities from nbutils package
from src.nbutils.misc import load_simulation_tree
from src.nbutils.master import *

# Importing individual local modules
from src import ansatz
from src import constraint
from src import createparam
from src import hamiltonian
from src import modules
from src import observable
from src import time_evolution_gate
from src import validator
from src import vqe
from src import zne

In [ ]:
PATH = "../experiment14[depol-time-evo-noisy_variousorderzne_tmax_20]/data/VQE"

In [ ]:
DATA_PATH = Path(
    PATH
)

final_states_dict = {}

for json_file in sorted(DATA_PATH.glob("*.json")):
    with open(json_file, "r") as f:
        data = json.load(f)

    final_states_dict[json_file.stem] = data["others"]["final_states"][0]

print(final_states_dict)

In [ ]:
final_states_dict.keys()
vqe_out_state_sample = state.from_json(final_states_dict["xy_ansatz_time_depol_p_0.001-0.005-0.001-0.001_tmax_20_run000_VQE"])
N_QUBIT = vqe_out_state_sample.get_qubit_count()
print(f"N_QUBIT = {N_QUBIT}")

In [ ]:
TARGET_OBSERVABLE = hamiltonian.create_ising_hamiltonian(N_QUBIT)
eigenvalues, _ = np.linalg.eigh(TARGET_OBSERVABLE.get_matrix().toarray())
exact_min_eigenvalue = np.min(eigenvalues)
print(f"Exact minimum eigenvalue: {exact_min_eigenvalue}")

In [ ]:
energies = []

for item in final_states_dict.keys():
    single_vqe_out_state = state.from_json(final_states_dict[item])

    single_inf_shot_vqe_est = TARGET_OBSERVABLE.get_expectation_value(
        single_vqe_out_state
    )

    print(
        f"Expectation value with infinite shots for {item}: "
        f"{single_inf_shot_vqe_est}"
    )

    energies.append(single_inf_shot_vqe_est)

inf_shot_vqe_est_mean = np.mean(energies)
inf_shot_vqe_est_std = np.std(energies)

print(f"Mean of infinite shot expectation values: {inf_shot_vqe_est_mean}")
print(f"Standard deviation of infinite shot expectation values: {inf_shot_vqe_est_std}")

In [ ]:
print(TARGET_OBSERVABLE)

In [ ]:
def estimate_expectation_shots(
    observable_hami,
    state,
    n_shots: int,
):
    """
    Shot-based expectation value estimation with variance.
    Splits total n_shots equally between X and Z measurement bases.
    """
    n = state.get_qubit_count()
    n_terms = observable_hami.get_term_count()

    # ----------------------------------------------------------
    # Parse Hamiltonian
    # ----------------------------------------------------------
    xx_terms = []
    zz_terms = []
    x_terms = []
    z_terms = []
    constant_energy = 0.0

    for i in range(n_terms):
        term = observable_hami.get_term(i)
        coeff = term.get_coef().real
        pauli_ids = term.get_pauli_id_list()
        pauli_qubits = term.get_index_list()

        # Handle pure Identity terms (if any)
        if len(pauli_ids) == 0 or all(p == 0 for p in pauli_ids):
            constant_energy += coeff
            continue

        if len(pauli_ids) == 2 and all(p == 1 for p in pauli_ids):
            xx_terms.append((coeff, pauli_qubits[0], pauli_qubits[1]))
        elif len(pauli_ids) == 2 and all(p == 3 for p in pauli_ids):
            zz_terms.append((coeff, pauli_qubits[0], pauli_qubits[1]))
        elif len(pauli_ids) == 1 and pauli_ids[0] == 1:
            x_terms.append((coeff, pauli_qubits[0]))
        elif len(pauli_ids) == 1 and pauli_ids[0] == 3:
            z_terms.append((coeff, pauli_qubits[0]))
        else:
            print(f"[WARNING] Term {i} skipped (unsupported Pauli type)")

    # ----------------------------------------------------------
    # Allocate Shot Budget
    # ----------------------------------------------------------
    # Split shots between the two bases needed
    has_x = bool(xx_terms or x_terms)
    has_z = bool(zz_terms or z_terms)
    
    # Simple equal split strategy
    shots_x = n_shots // 2 if (has_x and has_z) else (n_shots if has_x else 0)
    shots_z = n_shots - shots_x if (has_x and has_z) else (n_shots if has_z else 0)

    # ----------------------------------------------------------
    # X-basis measurement group
    # ----------------------------------------------------------
    x_shot_energies = np.zeros(shots_x)
    if shots_x > 0:
        # Clone state and rotate to X basis (Hadamard)
        x_state = state.copy()
        from qulacs import QuantumCircuit
        rot = QuantumCircuit(n)
        for q in range(n):
            rot.add_H_gate(q)
        rot.update_quantum_state(x_state)

        # Sampling: Qulacs returns an array of integers
        samples = x_state.sampling(shots_x)
        for shot_idx, bitstring in enumerate(samples):
            # Bit extraction: bitstring LSB is qubit 0
            e_shot = 0.0
            
            # Optimization: only extract bits for qubits we actually care about,
            # or pre-unpack up to max needed index.
            bits = [(bitstring >> i) & 1 for i in range(n)]
            xvals = [1 - 2 * b for b in bits]

            for coeff, qi, qj in xx_terms:
                e_shot += coeff * xvals[qi] * xvals[qj]
            for coeff, qi in x_terms:
                e_shot += coeff * xvals[qi]

            x_shot_energies[shot_idx] = e_shot

    # ----------------------------------------------------------
    # Z-basis measurement group
    # ----------------------------------------------------------
    z_shot_energies = np.zeros(shots_z)
    if shots_z > 0:
        samples = state.sampling(shots_z)
        for shot_idx, bitstring in enumerate(samples):
            e_shot = 0.0
            bits = [(bitstring >> i) & 1 for i in range(n)]
            zvals = [1 - 2 * b for b in bits]

            for coeff, qi, qj in zz_terms:
                e_shot += coeff * zvals[qi] * zvals[qj]
            for coeff, qi in z_terms:
                e_shot += coeff * zvals[qi]

            z_shot_energies[shot_idx] = e_shot

    # ----------------------------------------------------------
    # Mean and Variance Calculations
    # ----------------------------------------------------------
    mean_x = x_shot_energies.mean() if shots_x > 0 else 0.0
    mean_z = z_shot_energies.mean() if shots_z > 0 else 0.0
    
    # Combined mean includes the static identity offset
    mean_energy = mean_x + mean_z + constant_energy

    var_energy = 0.0
    # Var(Sample Mean) = Var(Shot Energies) / N_shots
    if shots_x > 1:
        var_energy += np.var(x_shot_energies, ddof=1) / shots_x
    if shots_z > 1:
        var_energy += np.var(z_shot_energies, ddof=1) / shots_z

    stderr = np.sqrt(var_energy)

    return mean_energy, var_energy, stderr

In [ ]:
energy, variance, stderr = estimate_expectation_shots(
    TARGET_OBSERVABLE,
    vqe_out_state_sample,
    n_shots=5000,
)

print(f"E      = {energy}")
print(f"Var(E) = {variance}")
print(f"SE     = {stderr}")

In [ ]:
shot_list = [10, 100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000, 200000, 500000, 1000000]

energies = []
variances = []
stderrs = []

for shots in shot_list:

    energy, variance, stderr = estimate_expectation_shots(
        TARGET_OBSERVABLE,
        vqe_out_state_sample,
        n_shots=shots,
    )

    energies.append(energy)
    variances.append(variance)
    stderrs.append(stderr)

# Plot energy with error bars
plt.figure(figsize=(6,4))
plt.errorbar(
    shot_list,
    energies,
    yerr=stderrs,
    marker="o",
    capsize=4,
    label="Estimated Energy with finite shots"
)
plt.axhline(
    inf_shot_vqe_est_mean,
    color="r",
    linestyle="-",
    label="Infinite-shot estimates over 10 trials"
)
plt.axhspan(
    inf_shot_vqe_est_mean - inf_shot_vqe_est_std,
    inf_shot_vqe_est_mean + inf_shot_vqe_est_std,
    alpha=0.2,
    color="r",
)
plt.axhline(y=exact_min_eigenvalue, color="g", linestyle="--", label="Exact minimum eigenvalue")
plt.xscale("log")
plt.xlabel("Shots")
plt.ylabel("Energy")
plt.title("Shot-based energy estimation")
#plt.grid(True)
plt.legend()
plt.show()

---

In [2]:
from quri_parts.core.operator import Operator, pauli_label, PAULI_IDENTITY
from quri_parts.core.state import quantum_state

op = Operator({
    pauli_label("Z0"): 0.25,
    pauli_label("Z1 Z2"): 2.0,
    pauli_label("X1 X2"): 0.5 + 0.25j,
    pauli_label("Z1 Y3"): 1.0j,
    pauli_label("Z2 Y3"): 1.5 + 0.5j,
    pauli_label("X1 Y3"): 2.0j,
    PAULI_IDENTITY: 3.0,
})

print("Operator:")
print(op)


state = quantum_state(4, bits=0b0101)
print("")
print("State:")
print(state)

Operator:
0.25*Z0 + 2.0*Z1 Z2 + (0.5+0.25j)*X1 X2 + 1j*Z1 Y3 + (1.5+0.5j)*Z2 Y3 + 2j*X1 Y3 + 3.0*I

State:
ComputationalBasisState(qubit_count=4, bits=0b101, phase=0π/2)


In [3]:
from quri_parts.core.estimator.sampling import create_sampling_estimator
from quri_parts.core.measurement import bitwise_commuting_pauli_measurement
from quri_parts.core.sampling.shots_allocator import create_proportional_shots_allocator
from quri_parts.qulacs.sampler import create_qulacs_vector_concurrent_sampler

total_shots = 5000
concurrent_sampler = create_qulacs_vector_concurrent_sampler()
measurement_factory = bitwise_commuting_pauli_measurement
shots_allocator = create_proportional_shots_allocator()

sampling_estimator = create_sampling_estimator(
    total_shots,
    concurrent_sampler,
    measurement_factory,
    shots_allocator
)